# Récupérer le corpus de discours présidentiels

**Ce notebook doit tourner sur une machine avec accès internet** 

Schéma vérifié à partir d'un échantillon réel du fichier. Les champs utiles sont:

* titre
* url
* domaine
* prononciation (date, format YYYY-MM-DD)
* intervenants (liste de `{nom, qualite, qualite_long}`)
* type_emetteur

```bash
pip install requests beautifulsoup4 tqdm
# optionnel, utilisé seulement en repli si le site bloque les requêtes "brutes" :
pip install playwright
playwright install chromium
```

Source : [Métadonnées des Discours publics de Vie-publique.fr, data.gouv.fr](https://www.data.gouv.fr/datasets/metadonnees-des-discours-publics-de-vie-publique-fr)

In [1]:
from datetime import datetime
import requests
from collections import Counter
from bs4 import BeautifulSoup
import time, re, os
from tqdm import tqdm
import glob

URL_METADONNEES = "https://www.data.gouv.fr/api/1/datasets/r/160fc156-2723-46ae-bbac-fd66f0b021e0"

## Configuration plage de dates

A modifier pour changer la période couverte.

In [2]:
# Plage de dates a recuperer 
DATE_DEBUT = datetime(2017, 5, 14)     # ex: datetime(1974, 1, 1) pour remonter a Giscard d'Estaing
DATE_FIN = datetime.now()             # ex: datetime(2017, 5, 14) pour s'arreter a la fin d'un mandat

#Limit du nombre de discours, à adapter suivant la config de la machine (300 fichier pour un RTX 2070 avec 16 Go DE RAM)
NOMBRE_MAX_DISCOURS = 300 

print("Periode:", DATE_DEBUT.date(), "->", DATE_FIN.date())
print("Duree:", (DATE_FIN - DATE_DEBUT).days, "jours (~", round((DATE_FIN - DATE_DEBUT).days / 365, 1), "ans)")

Periode: 2017-05-14 -> 2026-08-03
Duree: 3368 jours (~ 9.2 ans)


In [3]:
reponse = requests.get(URL_METADONNEES, timeout=60)
reponse.raise_for_status()
donnees = reponse.json()

liste_discours = donnees if isinstance(donnees, list) else donnees.get("data", donnees.get("discours", []))
print(len(liste_discours), "discours au total dans le fichier (toutes fonctions, toutes dates confondues)")

153473 discours au total dans le fichier (toutes fonctions, toutes dates confondues)


## Filtrage

`est_president()` vérifie deux choses : 
* Chaque intervenant du discours, en comparant sa `qualite` en égalité stricte (pas juste "contient") à "président de la république". exclut correctement les présidents étrangers rencontrés en entretien (ex. "Président de la République du Tchad", qui contiendrait la sous-chaîne mais n'est pas une égalité exacte)

* En filet de sécurité, les champs `domaine`/`type_emetteur` du document lui-même

In [4]:
def est_president(entree):
    for interv in (entree.get("intervenants") or []):
        qualite = (interv.get("qualite") or "").strip().lower()
        if qualite == "président de la république":
            return True
    domaine = (entree.get("domaine") or "").strip().lower()
    type_emetteur = (entree.get("type_emetteur") or "").strip().lower()
    return domaine == "président de la république" or type_emetteur == "président de la république"

def parser_date(entree):
    valeur = entree.get("prononciation")
    if not valeur:
        return None
    try:
        return datetime.strptime(valeur[:10], "%Y-%m-%d")
    except ValueError:
        return None

discours_filtres = []
for entree in liste_discours:
    if not est_president(entree):
        continue
    date_discours = parser_date(entree)
    if date_discours and DATE_DEBUT <= date_discours <= DATE_FIN:
        discours_filtres.append(entree)

print("=" * 60)
print(len(discours_filtres), "discours du President de la Republique entre", DATE_DEBUT.date(), "et", DATE_FIN.date())
print("=" * 60)
if discours_filtres:
    print("Exemple:", discours_filtres[0]["titre"][:100])

1482 discours du President de la Republique entre 2017-05-14 et 2026-08-03
Exemple: Déclaration de M. Emmanuel Macron, président de la République, sur la lutte contre les incidents de 


## Répartition par année

In [5]:
compte_par_annee = Counter(parser_date(e).year for e in discours_filtres if parser_date(e))
for annee in sorted(compte_par_annee):
    print(annee, ":", compte_par_annee[annee], "discours")

print("\nTotal:", len(discours_filtres), "discours sur", len(compte_par_annee), "annees")
if compte_par_annee:
    print("Moyenne:", round(len(discours_filtres) / len(compte_par_annee), 1), "discours/an")

2017 : 163 discours
2018 : 165 discours
2019 : 142 discours
2020 : 134 discours
2021 : 138 discours
2022 : 123 discours
2023 : 180 discours
2024 : 144 discours
2025 : 164 discours
2026 : 129 discours

Total: 1482 discours sur 10 annees
Moyenne: 148.2 discours/an


## Faisabilité locale: repère avant de lancer le téléchargement complet

Ordre de grandeur pour l'extraction d'entités/relations (cours GraphRAG, 1 appel LLM par discours) avec Qwen2.5-1.5B :

| Nombre de discours | RTX 2060 (laptop, 6 Go) | RTX 2070 (PC fixe, 8 Go) |
|---|---|---|
| ~50 | confortable, quelques minutes | confortable, plus rapide |
| ~150 | correct, 15-30 min | correct, 10-20 min |
| ~300 | limite haute recommandée, 30-60 min | correct, 20-40 min |
| 1000+ | déconseillé en local | à éviter aussi, plutôt échantillonner ou Colab |

La RTX 2070 (desktop, pas de throttling thermique, VRAM et bande passante mémoire plus élevées) encaisse mieux les gros volumes et permet aussi d'utiliser un modèle un peu plus grand (ex. Qwen2.5-3B) pour une meilleure qualité d'extraction si tu veux tenter au-delà de 300.

Le RAG (chunking + embeddings) supporte des volumes bien plus grands sans souci sur les deux machines (le coût n'est pas par appel LLM), donc la vraie contrainte vient du cours GraphRAG.

In [6]:
if len(discours_filtres) > NOMBRE_MAX_DISCOURS:
    print(len(discours_filtres), "discours trouves, echantillonnage a", NOMBRE_MAX_DISCOURS,
          "(repartis dans le temps, pas seulement les plus recents)")
    pas = len(discours_filtres) // NOMBRE_MAX_DISCOURS
    discours_filtres = discours_filtres[::pas][:NOMBRE_MAX_DISCOURS]
else:
    print("Volume sous le seuil (", NOMBRE_MAX_DISCOURS, ") — pas d'echantillonnage necessaire")

print(len(discours_filtres), "discours retenus pour le telechargement du texte integral")

1482 discours trouves, echantillonnage a 300 (repartis dans le temps, pas seulement les plus recents)
300 discours retenus pour le telechargement du texte integral


## Récupération du texte intégral depuis vie-publique.fr

In [ ]:
# DOSSIER_SORTIE = "./discours-presidents/"
DOSSIER_SORTIE = "./discours_"+str(DATE_DEBUT.date()) + "_"+str(DATE_FIN.date())
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

def nettoyer_nom_fichier(titre, date):
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", titre.lower())[:60].strip("-")
    return date + "_" + slug + ".txt"

# vie-publique.fr renvoie une page-coquille ("This website requires JS enabled
# and cookies") aux clients qui ne ressemblent pas a un vrai navigateur. On tente
# d'abord une session "navigateur realiste" (requests), et si ca ne suffit pas on
# bascule sur Playwright.
#
# Piege rencontre : la boucle asyncio du kernel Jupyter (geree par tornado) ne
# permet pas a Playwright de lancer le sous-processus Chromium -> erreur vide
# (NotImplementedError()). Solution : executer tout le code Playwright dans une
# boucle asyncio TOUTE NEUVE, sur un thread separe, independante de celle du
# kernel.

import asyncio
import sys
import threading

SIGNAUX_BLOCAGE = ("requires js", "enable javascript", "just a moment", "cf-browser-verification")

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
})
try:
    session.get("https://www.vie-publique.fr/", timeout=30)
except requests.RequestException:
    pass

def ressemble_a_un_blocage(texte):
    bas = texte.strip().lower()
    return len(bas) < 200 or any(signal in bas for signal in SIGNAUX_BLOCAGE)

def extraire_avec_requests(url):
    reponse = session.get(url, timeout=30)
    reponse.raise_for_status()
    soup = BeautifulSoup(reponse.text, "html.parser")
    corps = soup.find("div", class_=re.compile("field--name-field-texte|article-content|content", re.I))
    if corps is None:
        corps = soup.find("article") or soup.body
    return corps.get_text(separator="\n", strip=True) if corps else ""

def _extraire_corps_html(html):
    soup = BeautifulSoup(html, "html.parser")
    corps = soup.find("div", class_=re.compile("field--name-field-texte|article-content|content", re.I))
    if corps is None:
        corps = soup.find("article") or soup.body
    return corps.get_text(separator="\n", strip=True) if corps else ""

async def _telecharger_tout_async():
    from playwright.async_api import async_playwright

    erreurs = []
    async with async_playwright() as pw:
        navigateur = await pw.chromium.launch()

        async def extraire_avec_playwright(url):
            page = await navigateur.new_page()
            try:
                await page.goto(url, timeout=30000, wait_until="networkidle")
                await page.wait_for_timeout(1000)
                html = await page.content()
            finally:
                await page.close()
            return _extraire_corps_html(html)

        for entree in tqdm(discours_filtres):
            url = entree.get("url")
            titre = str(entree.get("titre", "sans-titre")).replace("\r", " ").replace("\n", " ").strip()
            date_str = str(entree.get("prononciation", "0000-00-00"))[:10]
            nom_fichier = nettoyer_nom_fichier(titre, date_str)
            chemin = os.path.join(DOSSIER_SORTIE, nom_fichier)

            if os.path.exists(chemin):
                continue
            if not url:
                erreurs.append((titre, "pas d'URL"))
                continue

            try:
                texte = extraire_avec_requests(url)
                if ressemble_a_un_blocage(texte):
                    texte = await extraire_avec_playwright(url)
                if len(texte) < 200:
                    erreurs.append((titre, "texte trop court (" + str(len(texte)) + " caracteres) - probable blocage anti-bot"))
                    continue
                with open(chemin, "w", encoding="utf-8") as f:
                    f.write("Titre: " + titre + "\nDate: " + date_str + "\nSource: " + url + "\n\n" + texte)
            except Exception as e:
                erreurs.append((titre, repr(e)))

            await asyncio.sleep(0.5)

        await navigateur.close()

    return erreurs

def telecharger_tout_thread_isole():
    """Lance _telecharger_tout_async() dans un thread avec sa PROPRE boucle
    asyncio, pour ne jamais toucher a celle du kernel Jupyter."""
    resultat = {}
    exception_thread = {}

    def cible():
        try:
            # Sous Windows, le kernel Jupyter force WindowsSelectorEventLoopPolicy,
            # qui ne supporte pas la creation de sous-processus (necessaire pour
            # lancer Chromium). On force ici la politique Proactor, qui le
            # supporte, uniquement dans ce thread dedie.
            if sys.platform == "win32":
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                resultat["erreurs"] = loop.run_until_complete(_telecharger_tout_async())
            finally:
                loop.close()
        except Exception as e:
            exception_thread["erreur"] = e

    t = threading.Thread(target=cible)
    t.start()
    t.join()

    if "erreur" in exception_thread:
        raise exception_thread["erreur"]
    return resultat.get("erreurs", [])

erreurs = telecharger_tout_thread_isole()

print(len(os.listdir(DOSSIER_SORTIE)), "fichiers dans", DOSSIER_SORTIE)
print(len(erreurs), "erreurs")
for titre, erreur in erreurs[:10]:
    print("-", titre, "->", erreur)

100%|██████████| 300/300 [17:14<00:00,  3.45s/it] 


300 fichiers dans ./discours_2017-05-14_2026-08-03
0 erreurs


## Vérification finale

In [10]:
fichiers = glob.glob(DOSSIER_SORTIE + "*.txt")
print(len(fichiers), "discours prets dans", DOSSIER_SORTIE)
if fichiers:
    with open(fichiers[0], encoding="utf-8") as f:
        print("\n--- Apercu du premier fichier ---")
        print(f.read()[:500])


0 discours prets dans ./discours_2017-05-14_2026-08-03
